# 11 — Judging pass: StrongREJECT + WildGuard on all nine stages

**Why this run exists.** Every cross-stage behavioural number currently rests on
the frozen regex classifier alone, because the original judge run used
`--scope confirmatory` and scored only the preregistered populations. On the 208
rows where all three scorers exist (quadrant C of M2 and M3), the regex
classifier agrees with WildGuard at **Cohen's kappa = 0.29**: it finds 1 refusal
in 104 rows where WildGuard finds 42, because it only matches templated refusals
("I cannot", "I'm sorry") while the models refuse these prompts in plain
language. So this pass is **load-bearing, not confirmatory**.

**No generation happens here.** The ~5,700 responses already exist; this only
runs the two judge models over them.

* wall clock **~80–110 min**, one T4, comfortably inside a session
* structure follows `08_final_gpu_run.ipynb`, the notebook with a proven run
* includes `bitsandbytes>=0.46.1` — **the exact missing package that made three
  earlier judge runs write complete-looking files containing no scores**

**Do not close the tab.** The run writes its output only at the end, so a
disconnect mid-pass loses the work (same as the 2-hour judge in notebook 08).

Run the cells in order. Cells 5, 6 and 7 are gates that stop early rather than
waste 90 minutes.


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone and pin the exact commit

Set `PINNED_COMMIT` to the SHA you pushed. It is validated by shape, so there is
no placeholder string to accidentally replace in two places.

In [ ]:
import os, re, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH   = 'main'

PINNED_COMMIT = '8e2bd49c454422cd32fa43eec547127d08e2b738'   # 40-char SHA of the commit with the dtype fix

assert re.fullmatch(r'[0-9a-f]{40}', PINNED_COMMIT), (
    f'PINNED_COMMIT must be a full 40-character SHA, got {PINNED_COMMIT!r}. '
    'Locally: git rev-parse HEAD')

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', PINNED_COMMIT], check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert head == PINNED_COMMIT, f'wrong commit: {head} != {PINNED_COMMIT}'
print('checked out', head)

## 3. Dependencies and GPU

**Deliberately NOT `pip install -r requirements.txt`.** Colab already ships
torch 2.11 / transformers 5.x, and the requirements file pins `torch>=2.2` with
no upper bound, so installing it upgrades torch *on disk underneath a running
Python process*. The next `import torch` then dies inside torch's own
initialisation with

    RuntimeError: Trying to override a python impl for DispatchKey.Autograd
                  on operator aten::dropout

which is the half-imported module being re-executed, and it masks the real
error. That is what happened on the first attempt at this notebook.

`--no-deps` on bitsandbytes keeps it from dragging a different torch in.

In [ ]:
!pip -q install -U --no-deps "bitsandbytes>=0.46.1" peft
# torchao (when present) breaks transformers on quantized/4-bit loads.
!pip uninstall -y torchao || true
!nvidia-smi

### 3b. GATE — imports are sane

If this fails, **Runtime -> Restart session**, then re-run from cell 1 and
**skip cell 3**. The packages are already installed; re-running the install is
what poisons the process.

In [ ]:
import torch, transformers
print('torch       ', torch.__version__)
print('transformers', transformers.__version__)
print('cuda        ', torch.cuda.is_available(), torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else '')
import bitsandbytes as bnb
print('bitsandbytes', bnb.__version__)
assert torch.cuda.is_available(), 'no GPU - switch the runtime type to T4'

## 4. Find the real Drive folder and bind `results/`

Copied from notebook 08: the `dpo_v2` folder can sit in `MyDrive`, behind a
shortcut, or on a shared drive. A candidate only counts if it actually contains
activations.

In [ ]:
import os, glob

candidates = ['/content/drive/MyDrive/dpo_v2']
candidates += sorted(glob.glob('/content/drive/.shortcut-targets-by-id/*/dpo_v2'))
candidates += sorted(glob.glob('/content/drive/Shareddrives/*/dpo_v2'))

real_root = None
for c in candidates:
    if glob.glob(os.path.join(c, 'results', 'behavioral_judges', '*.json')) or \
       glob.glob(os.path.join(c, 'results', 'activations', '*_final.npy')):
        real_root = c
        break
if real_root is None:
    raise SystemExit('No dpo_v2 folder with real results found. Checked:\n  '
                     + '\n  '.join(candidates))
os.environ['DPO_DRIVE_ROOT'] = real_root
print('DPO_DRIVE_ROOT =', real_root)

from src.colab_persist import bind, status_line
info = bind(persist_hf_cache=False)   # judges are ~10 GB; keep them off Drive
print(status_line(info))

## 5. GATE — Hugging Face access

Both judges are gated: `allenai/wildguard` is `gated: auto` and
`dsbowen/strong_reject` returns 401 even for metadata. Add a Colab secret named
**`HF_TOKEN`** (read scope) and enable it for this notebook, and accept the terms
on both model pages first:

* <https://huggingface.co/allenai/wildguard>  (gated: auto)
* <https://huggingface.co/google/gemma-2b>  (gated: MANUAL - the
  StrongREJECT evaluator is a PEFT adapter on this base, so access to
  the base is what actually matters)

Setting the env var too, so any `!python` subprocess inherits it (pattern from
notebook 08).

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login, whoami

_tok = userdata.get('HF_TOKEN')          # the SECRET NAME, not the token itself
assert _tok, 'Add a Colab secret named HF_TOKEN (read scope) and enable it here.'
os.environ['HF_TOKEN'] = _tok            # inherited by subprocesses
os.environ['HUGGING_FACE_HUB_TOKEN'] = _tok
login(token=_tok, add_to_git_credential=False)
print('authenticated as:', whoami()['name'])

## 6. GATE — judge preflight

Loads each judge, then unloads it. **If either prints `False`, stop here.** This
is the gate whose absence produced three 15 MB files containing no scores.

In [ ]:
from src.analysis.behavioral_judges import (
    LazyModelJudge, DEFAULT_STRONGREJECT_MODEL, DEFAULT_WILDGUARD_MODEL)

# Never hardcode the model ids here. `dsbowen/strong_reject` was removed from
# HuggingFace, and notebook 04b still carries that dead name; the module default
# is qylu4156/strongreject-15k-v1, which is what produced the existing scores.
print('strong_reject:', DEFAULT_STRONGREJECT_MODEL)
print('wildguard    :', DEFAULT_WILDGUARD_MODEL)

results = {}
for name, model_id in (('strong_reject', DEFAULT_STRONGREJECT_MODEL),
                       ('wildguard', DEFAULT_WILDGUARD_MODEL)):
    j = LazyModelJudge(name, model_id, load_4bit=True, allow_download=True)
    ok = j.try_load()
    results[name] = ok
    print(f'{name:14s} {ok}' + ('' if ok else f'   {j.load_error}'))
    if ok:
        j.unload()

assert all(results.values()), (
    'A judge failed to load. STOP — continuing would burn ~90 minutes and '
    'produce a file with no scores in it. Check bitsandbytes and the HF token.')
print('\npreflight OK')

## 7. GATE - rebuild the manifest, behaviour files only

Two things this fixes.

**The committed manifest is dead.** All 33 of its paths still say `v2_raw_*` /
`causal_ablation_v2_*`, from before those files were renamed to
`responses_654_*`. Every one points at a file that no longer exists. Notebook 08
rebuilds the manifest in-session for exactly this reason.

**No resume file needed.** Restricting the manifest to the nine un-intervened
behaviour files gives 5,886 rows, which is precisely the set that is unscored.
Scoring them fresh costs about the same as a resumed full run and removes the
75 MB Drive upload entirely. The output supplements the authoritative judge file
rather than replacing it - the cross-stage analysis reads only
`{stage}_behavior` conditions.


In [ ]:
import json
from pathlib import Path
from src.analysis.behavioral_judges import build_consolidated_from_results
from src.common.stages import ALL_STAGES

full = build_consolidated_from_results(
    "results", "results/manifests/consolidated_rebuilt.json")
print("rebuilt:", len(full["entries"]), "response files found by scanning results/")

beh = [e for e in full["entries"] if "responses_654_" in e["response_file"]]
assert len(beh) == len(ALL_STAGES), (
    "expected %d behaviour files, found %d - generations missing from "
    "results/behavioral_eval/" % (len(ALL_STAGES), len(beh)))

manifest = dict(full)
manifest["entries"] = beh
MANIFEST = Path("results/manifests/consolidated_behaviour_only.json")
MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

n_rows = 0
for e in beh:
    rows = json.load(open(e["response_file"], encoding="utf-8", errors="replace"))
    n_rows += len(rows)
    assert Path(e["binding_file"]).exists(), "missing binding for " + e["response_file"]
    print("  %-34s %5d rows" % (Path(e["response_file"]).name, len(rows)))

print()
print("total %d rows across %d stages" % (n_rows, len(beh)))
print("estimated ~%.0f min at ~83 rows/min for both judges" % (n_rows / 83))

## 8. The judging pass  (~70-90 min - do not close the tab)

Run in-process rather than through the CLI, because the **coverage contract** is
only exposed on the Python API. It makes the run fail loudly if any of the nine
behaviour conditions is still unscored at the end, instead of reporting success
with gaps.

`scope='all'` is essential: the default `confirmatory` scope is exactly what
excluded these rows in the first place.


In [ ]:
import time
from src.analysis.behavioral_judges import run_judges
from src.common.stages import ALL_STAGES

CONTRACT = {
    scorer: [(stage + "_behavior", None) for stage in ALL_STAGES]
    for scorer in ("strong_reject", "wildguard")
}
print("contract:", len(ALL_STAGES), "behaviour conditions x 2 judges")

t0 = time.time()
out = run_judges(
    str(MANIFEST),
    out_dir="results/behavioral_judges",
    run_live=True,
    scope="all",
    load_4bit=True,
    allow_download=True,
    fail_closed=True,
    coverage_contract=CONTRACT,
)
print()
print("wrote", out, " in %.1f min" % ((time.time() - t0) / 60))

## 9. Verify coverage actually improved

In [ ]:
import json
from collections import defaultdict

payload = json.load(open(out, encoding='utf-8', errors='replace'))
print('judge_status   :', payload['judge_status'])
print('carried forward:', payload.get('carried_forward'))
print('total records  :', len(payload['records']))

cov = defaultdict(lambda: defaultdict(int))
for r in payload['records']:
    c = str(r.get('condition', ''))
    if not c.endswith('_behavior'):
        continue
    cov[c]['n'] += 1
    for s in ('regex', 'strong_reject', 'wildguard'):
        if (r.get(s) or {}).get('judge_status') == 'scored':
            cov[c][s] += 1

print(f'\n{"condition":26s} {"rows":>6s} {"regex":>7s} {"SR":>7s} {"WG":>7s}')
for c in sorted(cov):
    v = cov[c]
    print(f'{c:26s} {v["n"]:6d} {v["regex"]:7d} {v["strong_reject"]:7d} {v["wildguard"]:7d}')

incomplete = [c for c, v in cov.items()
              if v['strong_reject'] < v['n'] or v['wildguard'] < v['n']]
assert not incomplete, f'still incomplete: {incomplete}'
print('\nAll nine behaviour conditions fully scored by both judges.')

## 10. Report back

Paste these three things into the analysis session:

1. the **filename** printed below,
2. the **coverage table** from cell 9,
3. the **wall clock** from cell 8.

The cross-stage matrix, the 2×2 contrasts and the selectivity results are then
recomputed under all three scorers, and the regex-only numbers become a
sensitivity row rather than the headline.

In [ ]:
import os
print('new judge file:')
!ls -la results/behavioral_judges/*.json | tail -3
print('\nDrive root:', os.environ.get('DPO_DRIVE_ROOT'))
print('commit    :', os.popen('git rev-parse HEAD').read().strip())